# Chapter 9: Machine Learning in Julia

**Duration: 3 hours**

## 9.1 The Julia ML Ecosystem

Julia has a growing ML ecosystem:
- **MLJ.jl** -- unified ML framework (like scikit-learn)
- **Flux.jl** -- differentiable programming / deep learning
- **Turing.jl** -- probabilistic programming
- **ScikitLearn.jl** -- wrapper for scikit-learn

## 9.2 MLJ.jl: Machine Learning Framework

In [ ]:
using Pkg
Pkg.add(["MLJ", "MLJDecisionTreeInterface", "MLJLinearModels"])

In [ ]:
using MLJ

# Load a built-in dataset
X, y = @load_iris
first(X, 5)

In [ ]:
# Train a decision tree classifier
Tree = @load DecisionTreeClassifier pkg=DecisionTree
tree = Tree(max_depth=3)

# Split data
train, test = partition(eachindex(y), 0.7, shuffle=true, rng=42)

# Create a machine and fit
mach = machine(tree, X, y)
fit!(mach, rows=train)

# Predict
ypred = predict_mode(mach, rows=test)

# Evaluate
println("Accuracy: $(accuracy(ypred, y[test]))")

In [ ]:
# Cross-validation
evaluate(tree, X, y,
    resampling=CV(nfolds=5, shuffle=true, rng=42),
    measure=[accuracy, log_loss]
)

## 9.3 Regression with MLJ

In [ ]:
using MLJ

# Generate synthetic regression data
X, y = make_regression(100, 3; noise=0.5, rng=42)

# Load linear regression
LinReg = @load LinearRegressor pkg=MLJLinearModels
model = LinReg()

mach = machine(model, X, y)
fit!(mach)

# Predictions
ypred = predict(mach)
println("Training RMSE: $(rmse(ypred, y))")

## 9.4 Model Tuning

In [ ]:
using MLJ

X, y = @load_iris

Tree = @load DecisionTreeClassifier pkg=DecisionTree
tree = Tree()

# Define hyperparameter range
r = range(tree, :max_depth, lower=1, upper=10)

# Grid search
tuned_tree = TunedModel(
    model=tree,
    ranges=[r],
    tuning=Grid(resolution=10),
    resampling=CV(nfolds=5),
    measure=accuracy
)

mach = machine(tuned_tree, X, y)
fit!(mach)

# Best model
println("Best max_depth: $(report(mach).best_model.max_depth)")

## 9.5 Pipelines

In [ ]:
using MLJ

X, y = @load_iris

# Build a pipeline: standardize → PCA → classifier
Tree = @load DecisionTreeClassifier pkg=DecisionTree

pipe = Standardizer() |> PCA(maxoutdim=2) |> Tree(max_depth=3)

mach = machine(pipe, X, y)
evaluate!(mach, resampling=CV(nfolds=5), measure=accuracy)

## 9.6 Introduction to Flux.jl

In [ ]:
using Pkg
Pkg.add("Flux")

In [ ]:
using Flux

# Define a simple neural network
model = Chain(
    Dense(4 => 16, relu),
    Dense(16 => 8, relu),
    Dense(8 => 3),
    softmax
)

println(model)
println("Parameters: $(sum(length, Flux.params(model)))")

In [ ]:
# Training loop example
using Flux
using Flux: onehotbatch, crossentropy

# Synthetic data for demonstration
X_train = randn(Float32, 4, 100)   # 4 features, 100 samples
y_labels = rand(1:3, 100)          # 3 classes
Y_train = onehotbatch(y_labels, 1:3)

model = Chain(
    Dense(4 => 16, relu),
    Dense(16 => 3),
    softmax
)

opt = Flux.setup(Adam(0.01), model)

# Training
for epoch in 1:100
    grads = Flux.gradient(model) do m
        crossentropy(m(X_train), Y_train)
    end
    Flux.update!(opt, model, grads[1])
    if epoch % 20 == 0
        l = crossentropy(model(X_train), Y_train)
        println("Epoch $epoch, Loss: $l")
    end
end

## 9.7 A Simple Regression with Flux

In [ ]:
using Flux

# Generate data: y = 2x + 1 + noise
x_data = reshape(Float32.(range(-2, 2, length=100)), 1, :)
y_data = 2f0 .* x_data .+ 1f0 .+ 0.2f0 .* randn(Float32, 1, 100)

# Simple linear model
model = Dense(1 => 1)

loss(m, x, y) = Flux.mse(m(x), y)
opt = Flux.setup(Adam(0.01), model)

for epoch in 1:200
    grads = Flux.gradient(m -> loss(m, x_data, y_data), model)
    Flux.update!(opt, model, grads[1])
end

println("Learned weight: $(model.weight), bias: $(model.bias)")
println("Expected: weight ≈ 2.0, bias ≈ 1.0")

## Exercises

In [ ]:
# Exercise 1: Load the iris dataset with MLJ. Train a KNN classifier
# and evaluate with 5-fold CV.
# Your code here

In [ ]:
# Exercise 2: Generate regression data. Compare LinearRegressor with
# DecisionTreeRegressor. Which has lower RMSE?
# Your code here

In [ ]:
# Exercise 3: Build an MLJ pipeline: Standardizer → RandomForestClassifier.
# Tune the number of trees.
# Your code here

In [ ]:
# Exercise 4: Train a Flux neural network to learn sin(x) on [-π, π].
# Plot the learned function vs true sin(x).
# Your code here

In [ ]:
# Exercise 5: Use Flux to build a simple autoencoder on random data.
# Your code here